# Document Q&A Chatbot (RAG)

Upload PDF, DOCX or TXT files and ask questions about them. The text is split into chunks, embedded with sentence-transformers and stored in a FAISS index; the most relevant chunks are passed to a small open LLM (TinyLlama) that answers only from those chunks. A Gradio chat window ties it together.

## How to run
- Open in Google Colab. Runtime type: **T4 GPU**.
- Run the cells from top to bottom.

_Rewritten from my original Colab notebook: `Bot (Docx).ipynb`._

In [ ]:
!pip install -q transformers accelerate sentence-transformers faiss-cpu gradio pypdf python-docx

## 1. Upload your documents

In [ ]:
from google.colab import files
uploaded = files.upload()   # choose one or more .pdf / .docx / .txt files

## 2. Read text from each file

In [ ]:
from pypdf import PdfReader
from docx import Document

def read_file(path):
    ext = path.lower().rsplit(".", 1)[-1]
    if ext == "pdf":
        return "\n".join(page.extract_text() or "" for page in PdfReader(path).pages)
    if ext == "docx":
        return "\n".join(p.text for p in Document(path).paragraphs if p.text.strip())
    if ext == "txt":
        with open(path, encoding="utf-8", errors="ignore") as f:
            return f.read()
    print("Skipping unsupported file:", path)
    return ""

docs = [{"source": name, "text": read_file(name)} for name in uploaded]
docs = [d for d in docs if d["text"].strip()]
print("Loaded", len(docs), "documents")

## 3. Split into overlapping chunks

In [ ]:
def chunk_text(text, chunk_size=200, overlap=40):
    words = text.split()
    chunks = []
    for start in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[start:start + chunk_size])
        if chunk:
            chunks.append(chunk)
    return chunks

chunks = []
for d in docs:
    for c in chunk_text(d["text"]):
        chunks.append({"source": d["source"], "text": c})
print("Total chunks:", len(chunks))

## 4. Embed chunks and build a FAISS index
With normalized vectors, inner product equals cosine similarity.

In [ ]:
import faiss
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
embeddings = embedder.encode([c["text"] for c in chunks], normalize_embeddings=True,
                             convert_to_numpy=True, show_progress_bar=True).astype("float32")
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

def retrieve(query, top_k=4):
    q = embedder.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    scores, ids = index.search(q, min(top_k, len(chunks)))
    return [{**chunks[i], "score": float(s)} for s, i in zip(scores[0], ids[0]) if i != -1]

## 5. Load a small chat model

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

llm_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(llm_name)
llm = AutoModelForCausalLM.from_pretrained(
    llm_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)

## 6. Answer a question from the retrieved context

In [ ]:
def answer(question):
    hits = retrieve(question)
    context = "\n\n".join(f"[{i}] ({h['source']}) {h['text']}" for i, h in enumerate(hits, 1))
    messages = [
        {"role": "system", "content": "Answer ONLY using the context. If the answer is not in the context, "
                                      "say: I don't know based on the uploaded documents."},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]
    input_ids = tokenizer.apply_chat_template(messages, add_generation_prompt=True,
                                              return_tensors="pt").to(llm.device)
    with torch.no_grad():
        output = llm.generate(input_ids, max_new_tokens=256, do_sample=False,
                              pad_token_id=tokenizer.eos_token_id)
    reply = tokenizer.decode(output[0][input_ids.shape[1]:], skip_special_tokens=True).strip()
    sources = "\n".join(f"- {h['source']} (score {h['score']:.2f})" for h in hits)
    return f"{reply}\n\nSources:\n{sources}"

print(answer("What is this document about?"))

## 7. Chat UI

In [ ]:
import gradio as gr

gr.ChatInterface(fn=lambda message, history: answer(message),
                 title="Chat with your documents",
                 description="Answers come only from the files you uploaded.").launch()